# Calibration, clinical utility and joint risk stratification


In [ ]:
# Shared configuration and Cox PH model specification.
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

import lifelines
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, proportional_hazard_test

try:
    from dcurves import dca as dcurves_dca
except ImportError as error:
    raise ImportError("Install dcurves>=1.1.7 for standard survival DCA.") from error

SHARED_NOTEBOOK = os.environ.get(
    "CHD_SHARED_NOTEBOOK", str(Path.cwd() / "00_shared_config.ipynb")
)
if not Path(SHARED_NOTEBOOK).exists():
    raise FileNotFoundError(
        f"Shared notebook not found: {SHARED_NOTEBOOK}. Place it beside this notebook or set CHD_SHARED_NOTEBOOK."
    )
get_ipython().run_line_magic("run", f'"{SHARED_NOTEBOOK}"')

SERVER_SENSITIVITY_OUTPUT_DIR = Path(os.environ.get("CHD_SENSITIVITY_OUTPUT_DIR", SENSITIVITY_OUTPUT_DIR))
RESULTS_OUTPUT_DIR = SERVER_SENSITIVITY_OUTPUT_DIR / "04_results"
FIGURE_DATA_DIR = RESULTS_OUTPUT_DIR / "figure_source_data"
TABLE_DIR = RESULTS_OUTPUT_DIR / "tables"
FIGURE_DIR = RESULTS_OUTPUT_DIR / "figures"
RESULT5_FINAL_DIR = RESULTS_OUTPUT_DIR / "result5_final_cox_calibration_dca_audit"
FIGURE_DATA_DIR = RESULT5_FINAL_DIR / "source_data"
TABLE_DIR = RESULT5_FINAL_DIR / "tables"
FIGURE_DIR = RESULT5_FINAL_DIR / "figures"
for directory in [RESULTS_OUTPUT_DIR, FIGURE_DATA_DIR, TABLE_DIR, FIGURE_DIR, RESULT5_FINAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

LOCKED_RESULT5_MODEL = "Cox PH"
RESULT5_MODEL_NORM = "Cox PH"
SAVED_MODEL_LABEL = "Cox PH"
LOCKED_MODEL_SELECTION_BASIS = (
    "Manuscript design lock: Result 5 evaluates the integrated Cox PH model; test performance was not used for selection"
)
PRIMARY_FULL_FEATURE_SET = "Clinical_plus_PRS_plus_NMR_score"
FEATURE_ORDER = ["Clinical_only", "Clinical_plus_PRS", "Clinical_plus_NMR_score", "Clinical_plus_PRS_plus_NMR_score"]
FEATURE_LABELS = {
    "Clinical_only": "Clinical", "Clinical_plus_PRS": "Clinical + PRS",
    "Clinical_plus_NMR_score": "Clinical + NMR_score",
    "Clinical_plus_PRS_plus_NMR_score": "Clinical + PRS + NMR_score",
}

def normalize_model(value):
    return "Cox PH" if "cox" in str(value).lower() else str(value)

print("Shared config:", SHARED_NOTEBOOK)
print("Locked Result 5 model:", RESULT5_MODEL_NORM)
print("Selection basis:", LOCKED_MODEL_SELECTION_BASIS)
print("Result 5 output:", RESULT5_FINAL_DIR)


In [ ]:
# 1. CNS-style plotting and provenance helpers
CNS = {
    "blue": "#3C5488",
    "red": "#E64B35",
    "green": "#00A087",
    "teal": "#4DBBD5",
    "orange": "#F39B7F",
    "purple": "#8491B4",
    "gray": "#6E6E6E",
    "light_gray": "#D9D9D9",
    "black": "#222222",
    "white": "#FFFFFF",
}
FEATURE_COLORS = {
    "Clinical_only": CNS["black"],
    "Clinical_plus_PRS": CNS["blue"],
    "Clinical_plus_NMR_score": CNS["green"],
    "Clinical_plus_PRS_plus_NMR_score": CNS["red"],
}

plt.rcParams.update({
    "figure.dpi": 170,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 13.0,
    "axes.titlesize": 15.0,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.0,
    "ytick.labelsize": 12.0,
    "legend.fontsize": 11.0,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

def clean_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis=grid_axis, color=CNS["light_gray"], linewidth=0.7, alpha=0.85)
    ax.tick_params(length=3.5, width=0.8, color=CNS["black"])

def save_figure(fig, stem):
    paths = []
    for extension in ["png", "pdf", "svg"]:
        path = FIGURE_DIR / f"{stem}.{extension}"
        fig.savefig(path, dpi=600, bbox_inches="tight", facecolor="white")
        paths.append(path)
    print("Saved:", stem)
    return paths

def quantile_ci(values, lower=0.025, upper=0.975):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return float(np.quantile(values, lower)), float(np.quantile(values, upper))

def weighted_quantile(values, weights, quantile):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[ok], weights[ok]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights) / np.sum(weights)
    return float(np.interp(float(quantile), cumulative, values))


In [ ]:
# Result 5 must consume the four locked Result 2 Cox artifacts.
# It must never silently refit prediction models.

(
    _df_train_check,
    _df_val_check,
    _df_test_check,
    _feature_sets_check,
    _manifest_check,
) = validate_pipeline_artifacts(load_frames=True)

_result5_required_features = [
    "Clinical_only",
    "Clinical_plus_PRS",
    "Clinical_plus_NMR_score",
    "Clinical_plus_PRS_plus_NMR_score",
]

_result5_locked_feature_sets = {
    name: _feature_sets_check[name]
    for name in _result5_required_features
}

load_all_risk_records(
    _result5_locked_feature_sets,
    ["Cox PH"],
)

log_progress(
    "Result5 preflight passed | "
    "four signed Result2 Cox risk artifacts loaded | no model refitting"
)


In [ ]:
# 2. Load immutable Stage 01 frames and saved individual risk scores
df_train, df_val, df_test, final_feature_sets, pipeline_manifest = validate_pipeline_artifacts(
    load_frames=True
)

missing_feature_sets = [name for name in FEATURE_ORDER if name not in final_feature_sets]
if missing_feature_sets:
    raise KeyError("Missing Stage 01 feature sets: " + ", ".join(missing_feature_sets))

selected_feature_sets = {name: final_feature_sets[name] for name in FEATURE_ORDER}
saved_risk_store = load_all_risk_records(selected_feature_sets, [SAVED_MODEL_LABEL])

def saved_risk_record(feature_set):
    exact_key = (MAIN_SCENARIO_NAME, feature_set, SAVED_MODEL_LABEL)
    if exact_key in saved_risk_store:
        return saved_risk_store[exact_key], SAVED_MODEL_LABEL
    for key, record in saved_risk_store.items():
        if (
            len(key) == 3
            and str(key[0]) == str(MAIN_SCENARIO_NAME)
            and str(key[1]) == str(feature_set)
            and normalize_model(key[2]) == RESULT5_MODEL_NORM
        ):
            return record, key[2]
    raise KeyError(f"Missing saved risk record for {feature_set} | {SAVED_MODEL_LABEL}")

raw_risk = {}
risk_audit_rows = []
test_frame_hash = dataframe_digest(df_test, ["Eid", "Time", "Event"])
train_frame_hash = dataframe_digest(df_train, ["Eid", "Time", "Event"])

for feature_set in FEATURE_ORDER:
    record, resolved_label = saved_risk_record(feature_set)
    train_risk = np.asarray(record["train_risk"], dtype=float).reshape(-1)
    test_risk = np.asarray(record["test_risk"], dtype=float).reshape(-1)
    if len(train_risk) != len(df_train) or len(test_risk) != len(df_test):
        raise RuntimeError(
            f"Risk/frame length mismatch for {feature_set}: "
            f"train={len(train_risk)}/{len(df_train)}, test={len(test_risk)}/{len(df_test)}"
        )
    if not np.isfinite(train_risk).all() or not np.isfinite(test_risk).all():
        raise ValueError(f"Non-finite saved risk score for {feature_set}")
    raw_risk[(feature_set, "train")] = train_risk
    raw_risk[(feature_set, "test")] = test_risk
    risk_audit_rows.append({
        "Feature_Set": feature_set,
        "Feature_Label": FEATURE_LABELS[feature_set],
        "Requested_Model": LOCKED_RESULT5_MODEL,
        "Resolved_Saved_Model": resolved_label,
        "Model_Selection_Basis": LOCKED_MODEL_SELECTION_BASIS,
        "Evaluation_Population": "locked test set",
        "Train_N": len(train_risk),
        "Test_N": len(test_risk),
        "Train_Frame_SHA256": train_frame_hash,
        "Test_Frame_SHA256": test_frame_hash,
        "Train_Risk_SHA256": vector_digest(train_risk),
        "Test_Risk_SHA256": vector_digest(test_risk),
    })

risk_audit = pd.DataFrame(risk_audit_rows)
risk_audit_path = FIGURE_DATA_DIR / "result5_final_saved_risk_and_model_lock_audit.csv"
risk_audit.to_csv(risk_audit_path, index=False)
display(risk_audit)

competing_event_column = os.environ.get("CHD_COMPETING_EVENT_COL", "Competing_Event").strip()
competing_risk_audit = pd.DataFrame([{
    "Primary_Estimand": "Cause-specific time to first incident CHD under right censoring",
    "Competing_Event_Column_Configured": competing_event_column,
    "Competing_Event_Column_Available": bool(
        competing_event_column and competing_event_column in df_test.columns
    ),
    "Required_Sensitivity_If_Available": (
        "Aalen-Johansen cumulative incidence and competing-risk DCA"
    ),
}])
competing_risk_audit.to_csv(
    FIGURE_DATA_DIR / "result5_final_competing_risk_estimand_audit.csv",
    index=False,
)
display(competing_risk_audit)


In [ ]:
# 3. Train-only absolute-risk calibration and censoring-aware calibration summaries
def fit_absolute_risk_calibrator(train_frame, train_risk, label):
    train_risk=np.asarray(train_risk,dtype=float).reshape(-1)
    center=float(np.mean(train_risk)); scale=float(np.std(train_risk,ddof=0))
    if not np.isfinite(scale) or scale<=0: raise ValueError(f"Degenerate training risk for {label}")
    calibration_frame=pd.DataFrame({"Time":train_frame["Time"].astype(float).to_numpy(),"Event":train_frame["Event"].astype(int).to_numpy(),"Risk_Score":(train_risk-center)/scale})
    calibrator=CoxPHFitter(penalizer=0.0,l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        calibrator.fit(calibration_frame,duration_col="Time",event_col="Event",show_progress=False)
    return {"label":label,"model":calibrator,"center":center,"scale":scale,"method":"train-fitted Cox recalibration of standardized raw model score"}

def predict_absolute_risk(calibrator, raw_risk, horizon):
    score=(np.asarray(raw_risk,dtype=float).reshape(-1)-calibrator["center"])/calibrator["scale"]
    survival=calibrator["model"].predict_survival_function(pd.DataFrame({"Risk_Score":score}),times=[float(horizon)])
    return np.clip(1.0-survival.iloc[0].to_numpy(dtype=float),0.0,1.0)

HORIZONS = [5.0, 10.0]
calibrators = {}
absolute_risk = {}
for feature_set in FEATURE_ORDER:
    calibrator = fit_absolute_risk_calibrator(
        df_train,
        raw_risk[(feature_set, "train")],
        f"{FEATURE_LABELS[feature_set]} | {RESULT5_MODEL_NORM}",
    )
    calibrators[feature_set] = calibrator
    for horizon in HORIZONS:
        prediction = predict_absolute_risk(
            calibrator,
            raw_risk[(feature_set, "test")],
            horizon,
        )
        if len(prediction) != len(df_test) or not np.isfinite(prediction).all():
            raise RuntimeError(f"Invalid {horizon:g}-year prediction for {feature_set}")
        absolute_risk[(feature_set, horizon)] = prediction

def km_risk_with_ci(frame, horizon, label="KM"):
    if len(frame) == 0:
        return np.nan, np.nan, np.nan
    km = KaplanMeierFitter(alpha=0.05)
    km.fit(
        frame["Time"].astype(float).to_numpy(),
        event_observed=frame["Event"].astype(int).to_numpy(),
        label=label,
    )
    survival = float(km.predict(float(horizon)))
    ci = km.confidence_interval_survival_function_
    eligible = ci.loc[ci.index.astype(float) <= float(horizon)]
    if eligible.empty:
        survival_lower = survival_upper = survival
    else:
        row = eligible.iloc[-1]
        lower_column = next((column for column in ci.columns if "lower" in str(column).lower()), ci.columns[0])
        upper_column = next((column for column in ci.columns if "upper" in str(column).lower()), ci.columns[-1])
        survival_lower = float(row[lower_column])
        survival_upper = float(row[upper_column])
    return (
        float(np.clip(1.0 - survival, 0.0, 1.0)),
        float(np.clip(1.0 - survival_upper, 0.0, 1.0)),
        float(np.clip(1.0 - survival_lower, 0.0, 1.0)),
    )

def calibration_by_quantile(eval_frame, prediction, horizon, groups=10):
    temporary = eval_frame[["Time", "Event"]].reset_index(drop=True).copy()
    temporary["Predicted_Risk"] = np.asarray(prediction, dtype=float).reshape(-1)
    temporary["Risk_Group"] = pd.qcut(
        temporary["Predicted_Risk"].rank(method="first"),
        q=int(groups),
        labels=False,
        duplicates="drop",
    ) + 1
    rows = []
    for group, data in temporary.groupby("Risk_Group", observed=True):
        observed, lower, upper = km_risk_with_ci(data, horizon, label=f"decile_{group}")
        rows.append({
            "Risk_Group": int(group),
            "N": int(len(data)),
            "Event_N": int(data["Event"].sum()),
            "Predicted_Risk": float(data["Predicted_Risk"].mean()),
            "Observed_Risk": observed,
            "Observed_Risk_Lower95": lower,
            "Observed_Risk_Upper95": upper,
        })
    return pd.DataFrame(rows).sort_values("Risk_Group")

def fixed_horizon_calibration_slope(eval_frame, prediction, horizon):
    prediction = np.clip(np.asarray(prediction, dtype=float), 1e-6, 1.0 - 1e-6)
    calibration_lp = np.log(-np.log(1.0 - prediction))
    original_time = eval_frame["Time"].astype(float).to_numpy()
    original_event = eval_frame["Event"].astype(int).to_numpy()
    # Administrative censoring makes the slope estimand match the stated horizon.
    frame = pd.DataFrame({
        "Time": np.minimum(original_time, float(horizon)),
        "Event": ((original_event == 1) & (original_time <= float(horizon))).astype(int),
        "Calibration_LP": calibration_lp,
    })
    model = CoxPHFitter(penalizer=0.0)
    model.fit(frame, duration_col="Time", event_col="Event", robust=True, show_progress=False)
    row = model.summary.loc["Calibration_LP"]
    return {
        "Calibration_Slope": float(row["coef"]),
        "Calibration_Slope_Lower95": float(row["coef lower 95%"]),
        "Calibration_Slope_Upper95": float(row["coef upper 95%"]),
    }

calibration_parts = []
calibration_summary_rows = []
for feature_set in FEATURE_ORDER:
    for horizon in HORIZONS:
        prediction = absolute_risk[(feature_set, horizon)]
        grouped = calibration_by_quantile(df_test, prediction, horizon, groups=10)
        grouped.insert(0, "Feature_Set", feature_set)
        grouped.insert(1, "Feature_Label", FEATURE_LABELS[feature_set])
        grouped.insert(2, "Model", RESULT5_MODEL_NORM)
        grouped.insert(3, "Split", "test")
        grouped.insert(4, "Horizon_Years", horizon)
        grouped["Test_Frame_SHA256"] = test_frame_hash
        calibration_parts.append(grouped)

        overall_observed, overall_lower, overall_upper = km_risk_with_ci(
            df_test, horizon, label="overall"
        )
        errors = np.abs(grouped["Observed_Risk"] - grouped["Predicted_Risk"])
        weights = grouped["N"].astype(float)
        slope = fixed_horizon_calibration_slope(df_test, prediction, horizon)
        calibration_summary_rows.append({
            "Feature_Set": feature_set,
            "Feature_Label": FEATURE_LABELS[feature_set],
            "Model": RESULT5_MODEL_NORM,
            "Split": "test",
            "Horizon_Years": horizon,
            "Mean_Predicted_Risk": float(np.mean(prediction)),
            "Overall_Observed_KM_Risk": overall_observed,
            "Overall_Observed_KM_Risk_Lower95": overall_lower,
            "Overall_Observed_KM_Risk_Upper95": overall_upper,
            "Calibration_In_The_Large_Risk_Difference": float(np.mean(prediction) - overall_observed),
            "Grouped_Weighted_Mean_Absolute_Error": float(np.average(errors, weights=weights)),
            "Grouped_Weighted_E50": weighted_quantile(errors, weights, 0.50),
            "Grouped_Weighted_E90": weighted_quantile(errors, weights, 0.90),
            **slope,
            "Observed_Risk_Method": "Kaplan-Meier with Greenwood 95% CI",
            "Test_Frame_SHA256": test_frame_hash,
        })

calibration_data = pd.concat(calibration_parts, ignore_index=True)
calibration_summary = pd.DataFrame(calibration_summary_rows)
calibration_data_path = FIGURE_DATA_DIR / "result5_final_calibration_by_decile_km_ci.csv"
calibration_summary_path = TABLE_DIR / "result5_final_censoring_aware_calibration_summary.csv"
calibration_data.to_csv(calibration_data_path, index=False)
calibration_summary.to_csv(calibration_summary_path, index=False)
print("Calibration source:", calibration_data_path)
print("Calibration summary:", calibration_summary_path)
display(calibration_summary)


In [ ]:
# 4. Standard survival DCA on the locked test set
DCA_THRESHOLDS = np.arange(0.005, 0.201, 0.005)
DCA_XLIM = {5.0: (1.0, 8.0), 10.0: (2.5, 15.0)}

def safe_dca_column(feature_set):
    return "model_" + re.sub(r"[^A-Za-z0-9_]+", "_", feature_set).strip("_").lower()

def standard_survival_dca(eval_frame, predictions, horizon):
    data = eval_frame[["Time", "Event"]].reset_index(drop=True).copy()
    model_columns = []
    label_map = {}
    feature_map = {}
    for feature_set, prediction in predictions.items():
        column = safe_dca_column(feature_set)
        data[column] = np.asarray(prediction, dtype=float).reshape(-1)
        model_columns.append(column)
        label_map[column] = FEATURE_LABELS[feature_set]
        feature_map[column] = feature_set

    result = dcurves_dca(
        data=data,
        outcome="Event",
        modelnames=model_columns,
        thresholds=np.asarray(DCA_THRESHOLDS, dtype=float),
        time_to_outcome_col="Time",
        time=float(horizon),
    ).copy()
    result.columns = [str(column).lower() for column in result.columns]
    rename = {}
    for candidate in ["model", "models", "label"]:
        if candidate in result.columns:
            rename[candidate] = "Strategy_Raw"
            break
    for candidate in ["threshold", "thresholds"]:
        if candidate in result.columns:
            rename[candidate] = "Threshold"
            break
    for candidate in ["net_benefit", "net benefit", "netbenefit"]:
        if candidate in result.columns:
            rename[candidate] = "Net_Benefit"
            break
    result = result.rename(columns=rename)
    required = {"Strategy_Raw", "Threshold", "Net_Benefit"}
    if not required.issubset(result.columns):
        raise RuntimeError("Unexpected dcurves output columns: " + ", ".join(result.columns))

    result["Strategy"] = result["Strategy_Raw"].replace(label_map).replace({
        "all": "Treat all",
        "none": "Treat none",
        "All": "Treat all",
        "None": "Treat none",
    })
    reverse_label = {value: key for key, value in label_map.items()}
    result["Feature_Set"] = result["Strategy"].map({
        label_map[column]: feature_map[column] for column in model_columns
    })
    result["Model"] = RESULT5_MODEL_NORM
    result["Split"] = "test"
    result["Horizon_Years"] = float(horizon)
    result["DCA_Method"] = "dcurves survival DCA; censoring handled internally; no external IPCW multiplication"
    result["Test_Frame_SHA256"] = test_frame_hash
    return result

dca_parts = []
for horizon in HORIZONS:
    predictions = {
        feature_set: absolute_risk[(feature_set, horizon)]
        for feature_set in FEATURE_ORDER
    }
    dca_parts.append(standard_survival_dca(df_test, predictions, horizon))

survival_dca_data = pd.concat(dca_parts, ignore_index=True)
dca_source_path = FIGURE_DATA_DIR / "result5_final_standard_survival_dca.csv"
survival_dca_data.to_csv(dca_source_path, index=False)

expected_strategies = set(FEATURE_LABELS.values()) | {"Treat all", "Treat none"}
for horizon in HORIZONS:
    observed = set(
        survival_dca_data.loc[
            survival_dca_data["Horizon_Years"].astype(float).eq(horizon), "Strategy"
        ].astype(str)
    )
    missing = sorted(expected_strategies.difference(observed))
    if missing:
        raise RuntimeError(f"Survival DCA is missing strategies at {horizon:g} years: {missing}")

print("Standard survival DCA source:", dca_source_path)
display(survival_dca_data.groupby(["Horizon_Years", "Strategy"]).size().rename("Rows"))


In [ ]:
# 5. Figure 5 plotting helpers
def calibration_for_horizon(horizon):
    return calibration_data.loc[
        (calibration_data["Feature_Set"] == PRIMARY_FULL_FEATURE_SET)
        & np.isclose(calibration_data["Horizon_Years"].astype(float), float(horizon))
    ].sort_values("Predicted_Risk").copy()

def dca_for_horizon(horizon):
    return survival_dca_data.loc[
        np.isclose(survival_dca_data["Horizon_Years"].astype(float), float(horizon))
    ].copy()

def plot_calibration(ax, calibration, title, horizon):
    data = calibration.sort_values("Predicted_Risk").copy()
    x = data["Predicted_Risk"].to_numpy(dtype=float) * 100
    y = data["Observed_Risk"].to_numpy(dtype=float) * 100
    lower = data["Observed_Risk_Lower95"].to_numpy(dtype=float) * 100
    upper = data["Observed_Risk_Upper95"].to_numpy(dtype=float) * 100
    axis_max = max(1.0, float(np.nanmax(np.r_[x, upper])) * 1.15)
    ax.plot([0, axis_max], [0, axis_max], color=CNS["gray"], linestyle="--", linewidth=1.0, label="Ideal")
    ax.errorbar(
        x, y,
        yerr=np.vstack([np.maximum(0, y - lower), np.maximum(0, upper - y)]),
        fmt="o-", color=CNS["red"], ecolor=CNS["red"],
        linewidth=1.6, elinewidth=0.9, capsize=2.5,
        markersize=5.6, markeredgecolor="white", markeredgewidth=0.6,
        label="Observed KM risk (95% CI)",
    )
    ax.set_xlim(0, axis_max)
    ax.set_ylim(0, axis_max)
    ax.set_xlabel(f"Predicted {horizon:g}-year CHD risk (%)")
    ax.set_ylabel(f"Observed {horizon:g}-year CHD risk (%)")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "both")
    ax.legend(frameon=False, loc="upper left")

def plot_observed_predicted(ax, calibration, title, horizon):
    data = calibration.sort_values("Predicted_Risk").reset_index(drop=True).copy()
    x = np.arange(1, len(data) + 1)
    observed = data["Observed_Risk"].to_numpy(dtype=float) * 100
    predicted = data["Predicted_Risk"].to_numpy(dtype=float) * 100
    lower = data["Observed_Risk_Lower95"].to_numpy(dtype=float) * 100
    upper = data["Observed_Risk_Upper95"].to_numpy(dtype=float) * 100
    ax.errorbar(
        x, observed,
        yerr=np.vstack([np.maximum(0, observed - lower), np.maximum(0, upper - observed)]),
        fmt="o-", color=CNS["red"], ecolor=CNS["red"],
        linewidth=1.7, elinewidth=0.8, capsize=2.5,
        markersize=5.3, label="Observed KM risk (95% CI)",
    )
    ax.plot(x, predicted, "s-", color=CNS["blue"], linewidth=1.5, markersize=4.8, label="Predicted")
    ax.set_xticks(x)
    ax.set_xlabel("Predicted-risk decile")
    ax.set_ylabel(f"{horizon:g}-year CHD risk (%)")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "y")
    ax.legend(frameon=False, loc="upper left")

def plot_dca(ax, dca, title, horizon):
    order = list(FEATURE_LABELS.values()) + ["Treat all", "Treat none"]
    label_to_feature = {label: feature for feature, label in FEATURE_LABELS.items()}
    for strategy in order:
        data = dca.loc[dca["Strategy"].astype(str).eq(strategy)].sort_values("Threshold")
        if data.empty:
            continue
        if strategy == "Treat all":
            color, linestyle, width = CNS["gray"], "--", 1.1
        elif strategy == "Treat none":
            color, linestyle, width = CNS["gray"], ":", 1.1
        else:
            feature = label_to_feature[strategy]
            color = FEATURE_COLORS[feature]
            linestyle = "-"
            width = 2.2 if feature == PRIMARY_FULL_FEATURE_SET else 1.5
        ax.plot(
            data["Threshold"].astype(float) * 100,
            data["Net_Benefit"].astype(float),
            color=color, linestyle=linestyle, linewidth=width, label=strategy,
        )
    ax.set_xlim(*DCA_XLIM[float(horizon)])
    ax.set_xlabel(f"{horizon:g}-year risk threshold (%)")
    ax.set_ylabel("Net benefit")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "both")
    ax.legend(frameon=False, loc="best")

def plot_delta_net_benefit(ax, dca, title, horizon):
    model_data = dca.loc[dca["Feature_Set"].notna()].copy()
    pivot = model_data.pivot_table(
        index="Threshold", columns="Feature_Set", values="Net_Benefit", aggfunc="mean"
    ).sort_index()
    if "Clinical_only" not in pivot.columns:
        raise RuntimeError("Clinical DCA reference is unavailable.")
    for feature_set in FEATURE_ORDER[1:]:
        if feature_set not in pivot.columns:
            raise RuntimeError(f"DCA feature is unavailable: {feature_set}")
        delta = pivot[feature_set] - pivot["Clinical_only"]
        ax.plot(
            pivot.index.astype(float) * 100,
            delta,
            color=FEATURE_COLORS[feature_set],
            linewidth=2.2 if feature_set == PRIMARY_FULL_FEATURE_SET else 1.5,
            label=FEATURE_LABELS[feature_set],
        )
    ax.axhline(0, color=CNS["gray"], linestyle=":", linewidth=1.0)
    ax.set_xlim(*DCA_XLIM[float(horizon)])
    ax.set_xlabel(f"{horizon:g}-year risk threshold (%)")
    ax.set_ylabel("Difference in net benefit relative to Clinical")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "both")
    ax.legend(frameon=False, loc="best")

def save_single_panel(plotter, data_args, stem, figsize=(7.0, 5.4)):
    fig, ax = plt.subplots(figsize=figsize)
    plotter(ax, *data_args)
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()


In [ ]:
# Figure 5: separate 10-year calibration and clinical-utility panels
calibration_10y = calibration_for_horizon(10.0)
dca_10y = dca_for_horizon(10.0)

save_single_panel(
    plot_calibration,
    (calibration_10y, f"Figure 5A. 10-year calibration ({RESULT5_MODEL_NORM})", 10.0),
    "figure5A_10y_calibration_km_ci_cns_final",
)
save_single_panel(
    plot_observed_predicted,
    (calibration_10y, f"Figure 5B. 10-year observed versus predicted risk ({RESULT5_MODEL_NORM})", 10.0),
    "figure5B_10y_observed_vs_predicted_km_ci_cns_final",
    figsize=(7.4, 5.3),
)
save_single_panel(
    plot_dca,
    (dca_10y, f"Figure 5C. 10-year survival decision curve analysis ({RESULT5_MODEL_NORM})", 10.0),
    "figure5C_10y_standard_survival_dca_cns_final",
    figsize=(7.6, 5.5),
)
save_single_panel(
    plot_delta_net_benefit,
    (dca_10y, f"Figure 5D. Incremental clinical utility ({RESULT5_MODEL_NORM})", 10.0),
    "figure5D_10y_delta_net_benefit_survival_dca_cns_final",
    figsize=(7.6, 5.5),
)


In [ ]:
# Supplementary Figure 5: separate 5-year panels
calibration_5y = calibration_for_horizon(5.0)
dca_5y = dca_for_horizon(5.0)

save_single_panel(
    plot_calibration,
    (calibration_5y, f"Supplementary Figure 5A. 5-year calibration ({RESULT5_MODEL_NORM})", 5.0),
    "supplementary_figure5A_5y_calibration_km_ci_cns_final",
)
save_single_panel(
    plot_observed_predicted,
    (calibration_5y, f"Supplementary Figure 5B. 5-year observed versus predicted risk ({RESULT5_MODEL_NORM})", 5.0),
    "supplementary_figure5B_5y_observed_vs_predicted_km_ci_cns_final",
    figsize=(7.4, 5.3),
)
save_single_panel(
    plot_dca,
    (dca_5y, f"Supplementary Figure 5C. 5-year survival decision curve analysis ({RESULT5_MODEL_NORM})", 5.0),
    "supplementary_figure5C_5y_standard_survival_dca_cns_final",
    figsize=(7.6, 5.5),
)
save_single_panel(
    plot_delta_net_benefit,
    (dca_5y, f"Supplementary Figure 5D. 5-year incremental clinical utility ({RESULT5_MODEL_NORM})", 5.0),
    "supplementary_figure5D_5y_delta_net_benefit_survival_dca_cns_final",
    figsize=(7.6, 5.5),
)


In [ ]:
# 6. Rebuild all Figure 6 joint-risk data from one locked test population
PRS_ORDER = ["PRS_low", "PRS_mid", "PRS_high"]
NMR_ORDER = ["NMR_low", "NMR_mid", "NMR_high"]
PRS_LABELS = {"PRS_low": "Low PRS", "PRS_mid": "Intermediate PRS", "PRS_high": "High PRS"}
NMR_LABELS = {"NMR_low": "Low NMR_score", "NMR_mid": "Intermediate NMR_score", "NMR_high": "High NMR_score"}
GROUP_ORDER = [f"{prs}__{nmr}" for prs in PRS_ORDER for nmr in NMR_ORDER]
GROUP_LABELS = {
    f"{prs}__{nmr}": f"{PRS_LABELS[prs]} / {NMR_LABELS[nmr]}"
    for prs in PRS_ORDER for nmr in NMR_ORDER
}

def tertile_edges(train_series, label):
    edges = pd.to_numeric(train_series, errors="coerce").quantile([1 / 3, 2 / 3]).to_numpy(dtype=float)
    if len(edges) != 2 or not np.isfinite(edges).all() or len(np.unique(edges)) != 2:
        raise ValueError(f"Invalid {label} tertile edges: {edges}")
    return edges

def assign_tertiles(values, edges, prefix):
    return pd.cut(
        pd.to_numeric(values, errors="coerce"),
        bins=[-np.inf, float(edges[0]), float(edges[1]), np.inf],
        labels=[f"{prefix}_low", f"{prefix}_mid", f"{prefix}_high"],
        include_lowest=True,
    )

prs_edges = tertile_edges(df_train[PRS_COL], "PRS")
nmr_edges = tertile_edges(df_train["NMR_score"], "NMR_score")

joint_test = df_test.copy().reset_index(drop=True)
joint_test["Test_Row_Index"] = np.arange(len(joint_test), dtype=int)
joint_test["PRS_Tertile"] = assign_tertiles(joint_test[PRS_COL], prs_edges, "PRS")
joint_test["NMR_Tertile"] = assign_tertiles(joint_test["NMR_score"], nmr_edges, "NMR")
complete_mask = joint_test["PRS_Tertile"].notna() & joint_test["NMR_Tertile"].notna()
joint_test = joint_test.loc[complete_mask].copy().reset_index(drop=True)
joint_test["Group_Clean"] = (
    joint_test["PRS_Tertile"].astype(str) + "__" + joint_test["NMR_Tertile"].astype(str)
)
if not joint_test["Group_Clean"].isin(GROUP_ORDER).all():
    raise RuntimeError("Unexpected joint PRS/NMR group after complete-case filtering.")
joint_test["Group_Label"] = joint_test["Group_Clean"].map(GROUP_LABELS)

integrated_prediction = {
    horizon: absolute_risk[(PRIMARY_FULL_FEATURE_SET, horizon)]
    for horizon in HORIZONS
}

joint_absolute_rows = []
joint_km_rows = []
for group in GROUP_ORDER:
    group_frame = joint_test.loc[joint_test["Group_Clean"].eq(group)].copy()
    if group_frame.empty:
        raise RuntimeError(f"Empty joint group: {group}")
    row_index = group_frame["Test_Row_Index"].to_numpy(dtype=int)
    for horizon in HORIZONS:
        observed, lower, upper = km_risk_with_ci(group_frame, horizon, label=group)
        joint_absolute_rows.append({
            "Evaluation_Population": "test complete cases for PRS and NMR_score",
            "Group_Clean": group,
            "Group_Label": GROUP_LABELS[group],
            "PRS_Tertile": group.split("__")[0],
            "NMR_Tertile": group.split("__")[1],
            "Horizon_Years": horizon,
            "N": int(len(group_frame)),
            "Event_N": int(group_frame["Event"].sum()),
            "Observed_KM_Risk": observed,
            "Observed_KM_Risk_Lower95": lower,
            "Observed_KM_Risk_Upper95": upper,
            "Mean_Predicted_Risk": float(np.mean(integrated_prediction[horizon][row_index])),
            "Prediction_Model": RESULT5_MODEL_NORM,
            "Feature_Set": PRIMARY_FULL_FEATURE_SET,
            "Test_Frame_SHA256": test_frame_hash,
        })

    km = KaplanMeierFitter()
    km.fit(
        group_frame["Time"].astype(float),
        event_observed=group_frame["Event"].astype(int),
        label=group,
    )
    survival = km.survival_function_[group]
    for time_point, survival_probability in survival.items():
        if float(time_point) <= 10.0:
            joint_km_rows.append({
                "Evaluation_Population": "test complete cases for PRS and NMR_score",
                "Group_Clean": group,
                "Group_Label": GROUP_LABELS[group],
                "Time": float(time_point),
                "Survival": float(survival_probability),
                "Cumulative_CHD_Risk": float(1.0 - survival_probability),
                "N": int(len(group_frame)),
                "Event_N": int(group_frame["Event"].sum()),
                "Test_Frame_SHA256": test_frame_hash,
            })

joint_absolute = pd.DataFrame(joint_absolute_rows)
joint_km = pd.DataFrame(joint_km_rows)
joint_absolute_path = TABLE_DIR / "result5_final_joint_prs_nmr_observed_predicted_risk.csv"
joint_km_path = FIGURE_DATA_DIR / "result5_final_joint_prs_nmr_km_curve.csv"
joint_absolute.to_csv(joint_absolute_path, index=False)
joint_km.to_csv(joint_km_path, index=False)

missing_n = len(df_test) - len(joint_test)
joint_population_audit = pd.DataFrame([{
    "Locked_Test_N": int(len(df_test)),
    "Joint_Complete_Case_N": int(len(joint_test)),
    "Excluded_Missing_PRS_or_NMR_score_N": int(missing_n),
    "Excluded_Percent": 100.0 * missing_n / len(df_test),
    "PRS_Tertile_Cut_1": float(prs_edges[0]),
    "PRS_Tertile_Cut_2": float(prs_edges[1]),
    "NMR_Tertile_Cut_1": float(nmr_edges[0]),
    "NMR_Tertile_Cut_2": float(nmr_edges[1]),
    "Cutpoint_Source": "training set",
    "Test_Frame_SHA256": test_frame_hash,
}])
joint_population_audit.to_csv(
    FIGURE_DATA_DIR / "result5_final_joint_population_and_cutpoint_audit.csv",
    index=False,
)
display(joint_population_audit)


In [ ]:
# 7. Specification-locked adjusted-HR association model and PH diagnostics
HR_REFERENCE_GROUP = "PRS_low__NMR_low"
HR_FORBIDDEN_CONTINUOUS = {str(PRS_COL), "NMR_score"}
HR_ASSOCIATION_PENALIZER = 0.0

clinical_requested = list(final_feature_sets["Clinical_only"])
missing_clinical_train = [column for column in clinical_requested if column not in df_train.columns]
missing_clinical_test = [column for column in clinical_requested if column not in df_test.columns]
if missing_clinical_train or missing_clinical_test:
    raise KeyError(
        "Prespecified clinical adjustment variables are missing; refusing silent deletion. "
        f"train={missing_clinical_train}, test={missing_clinical_test}"
    )
if HR_FORBIDDEN_CONTINUOUS.intersection(clinical_requested):
    raise RuntimeError("Continuous PRS/NMR_score leaked into Clinical_only.")

X_clinical_train, X_clinical_val, X_clinical_test, clinical_actual = prepare_X_train_val_test(
    df_train,
    df_val,
    df_test,
    clinical_requested,
    scale=True,
)
clinical_actual = list(clinical_actual)
forbidden_actual = [
    column for column in clinical_actual
    if column in HR_FORBIDDEN_CONTINUOUS
    or str(column).startswith(str(PRS_COL) + "_")
    or str(column).startswith("NMR_score_")
]
if forbidden_actual:
    raise RuntimeError("Forbidden PRS/NMR terms entered the HR design: " + ", ".join(forbidden_actual))

test_rows = joint_test["Test_Row_Index"].to_numpy(dtype=int)
X_joint_clinical = X_clinical_test.iloc[test_rows].reset_index(drop=True).astype(float)
joint_category = pd.Categorical(joint_test["Group_Clean"], categories=GROUP_ORDER, ordered=True)
joint_dummies = pd.get_dummies(joint_category, prefix="Joint", drop_first=False, dtype=float)
reference_column = f"Joint_{HR_REFERENCE_GROUP}"
if reference_column not in joint_dummies.columns:
    raise RuntimeError(f"HR reference group is absent: {HR_REFERENCE_GROUP}")
joint_dummies = joint_dummies.drop(columns=[reference_column]).reset_index(drop=True)

hr_design = pd.concat(
    [
        joint_test[["Time", "Event"]].astype(float).reset_index(drop=True),
        X_joint_clinical,
        joint_dummies,
    ],
    axis=1,
)
if hr_design.columns.duplicated().any():
    duplicates = hr_design.columns[hr_design.columns.duplicated()].tolist()
    raise RuntimeError(f"Duplicated HR design columns: {duplicates}")
if len(hr_design) != len(joint_test):
    raise RuntimeError("HR design and joint test population are misaligned.")

hr_design_hash = dataframe_digest(hr_design, list(hr_design.columns))
hr_specification = {
    "formula": "Surv(Time, Event) ~ joint PRS-NMR group + prespecified clinical covariates",
    "reference_group": HR_REFERENCE_GROUP,
    "evaluation_population": "locked test-set PRS/NMR complete cases",
    "clinical_requested": clinical_requested,
    "clinical_design_columns": clinical_actual,
    "joint_dummy_columns": list(joint_dummies.columns),
    "continuous_prs_adjusted": False,
    "continuous_nmr_score_adjusted": False,
    "penalizer": HR_ASSOCIATION_PENALIZER,
    "prs_tertile_edges": prs_edges.tolist(),
    "nmr_tertile_edges": nmr_edges.tolist(),
    "test_frame_sha256": test_frame_hash,
    "hr_design_sha256": hr_design_hash,
    "n_test_complete_case": int(len(joint_test)),
    "lifelines_version": lifelines.__version__,
    "python_version": platform.python_version(),
}
hr_signature = hashlib.sha256(
    json.dumps(hr_specification, sort_keys=True).encode("utf-8")
).hexdigest()

hr_path = TABLE_DIR / "result5_final_joint_prs_nmr_adjusted_hr.csv"
hr_ph_path = TABLE_DIR / "result5_final_joint_prs_nmr_ph_diagnostics.csv"
hr_spec_path = TABLE_DIR / "result5_final_joint_prs_nmr_hr_specification.json"

cache_matches = False
if hr_path.exists() and hr_ph_path.exists() and hr_spec_path.exists():
    with hr_spec_path.open("r", encoding="utf-8") as handle:
        saved_specification = json.load(handle)
    cache_matches = saved_specification.get("signature") == hr_signature

if cache_matches:
    print("Loading specification-matched Figure 6C association cache.")
    joint_hr = pd.read_csv(hr_path)
    ph_diagnostics = pd.read_csv(hr_ph_path)
else:
    print("Fitting one small Figure 6 association Cox model; prediction models remain untouched.")
    association_cox = CoxPHFitter(penalizer=HR_ASSOCIATION_PENALIZER, l1_ratio=0.0)
    association_cox.fit(hr_design, duration_col="Time", event_col="Event", robust=True, show_progress=False)
    summary = association_cox.summary.reset_index()
    variable_column = next(
        (column for column in ["covariate", "index", "variable"] if column in summary.columns),
        summary.columns[0],
    )
    summary = summary.rename(columns={variable_column: "Variable"})
    summary = summary.loc[summary["Variable"].astype(str).str.startswith("Joint_")].copy()
    joint_hr = pd.DataFrame({
        "Reference_Group": HR_REFERENCE_GROUP,
        "Evaluation_Population": "test",
        "Variable": summary["Variable"].astype(str),
        "Group_Clean": summary["Variable"].astype(str).str.replace("Joint_", "", regex=False),
        "HR": np.exp(pd.to_numeric(summary["coef"], errors="coerce")),
        "HR_CI_Lower": np.exp(pd.to_numeric(summary["coef lower 95%"], errors="coerce")),
        "HR_CI_Upper": np.exp(pd.to_numeric(summary["coef upper 95%"], errors="coerce")),
        "P_Value": pd.to_numeric(summary["p"], errors="coerce"),
        "Association_Model": "joint-group indicators + clinical covariates",
        "Continuous_PRS_Adjusted": False,
        "Continuous_NMR_score_Adjusted": False,
        "Model_Specification_SHA256": hr_signature,
    })
    ph_test = proportional_hazard_test(
        association_cox,
        hr_design,
        time_transform="rank",
    )
    ph_diagnostics = ph_test.summary.reset_index().rename(columns={"index": "Term"})
    ph_diagnostics["Model_Specification_SHA256"] = hr_signature
    ph_diagnostics["Diagnostic"] = "Scaled Schoenfeld residual test; rank time transform"
    joint_hr.to_csv(hr_path, index=False)
    with hr_spec_path.open("w", encoding="utf-8") as handle:
        json.dump({"signature": hr_signature, **hr_specification}, handle, indent=2)

# Multiple-testing audit for term-wise proportional-hazards diagnostics.
ph_diagnostics["p"] = pd.to_numeric(
    ph_diagnostics["p"],
    errors="coerce",
)

ph_test_n = int(ph_diagnostics["p"].notna().sum())

ph_diagnostics["P_Bonferroni"] = np.minimum(
    ph_diagnostics["p"] * ph_test_n,
    1.0,
)

ph_diagnostics["PH_Flag_Raw_0.05"] = (
    ph_diagnostics["p"] < 0.05
)

ph_diagnostics["PH_Flag_Bonferroni"] = (
    ph_diagnostics["P_Bonferroni"] < 0.05
)

ph_diagnostics["PH_Test_N"] = ph_test_n

# Save the complete term-wise PH diagnostics.
ph_diagnostics.to_csv(
    hr_ph_path,
    index=False,
)

print(
    "PH diagnostics:",
    f"tests={ph_test_n}",
    f"raw_flags={int(ph_diagnostics['PH_Flag_Raw_0.05'].sum())}",
    f"Bonferroni_flags={int(ph_diagnostics['PH_Flag_Bonferroni'].sum())}",
)

display(
    ph_diagnostics
    .sort_values("p")
    .reset_index(drop=True)
)

reference_row = pd.DataFrame([{
    "Reference_Group": HR_REFERENCE_GROUP,
    "Evaluation_Population": "test",
    "Variable": "Reference",
    "Group_Clean": HR_REFERENCE_GROUP,
    "HR": 1.0,
    "HR_CI_Lower": 1.0,
    "HR_CI_Upper": 1.0,
    "P_Value": np.nan,
    "Association_Model": "joint-group indicators + clinical covariates",
    "Continuous_PRS_Adjusted": False,
    "Continuous_NMR_score_Adjusted": False,
    "Model_Specification_SHA256": hr_signature,
}])
joint_hr_plot = pd.concat([reference_row, joint_hr], ignore_index=True)
joint_hr_plot["Group_Label"] = joint_hr_plot["Group_Clean"].map(GROUP_LABELS)
joint_hr_plot["Group_Clean"] = pd.Categorical(
    joint_hr_plot["Group_Clean"], GROUP_ORDER, ordered=True
)
joint_hr_plot = joint_hr_plot.sort_values("Group_Clean").reset_index(drop=True)
if set(joint_hr_plot["Group_Clean"].astype(str)) != set(GROUP_ORDER):
    raise RuntimeError("Adjusted-HR output does not contain exactly nine joint groups.")

covariate_audit = pd.DataFrame(
    [{"Term": column, "Role": "clinical adjustment", "Included": True} for column in clinical_actual]
    + [{"Term": column, "Role": "joint-group contrast", "Included": True} for column in joint_dummies.columns]
    + [{"Term": column, "Role": "forbidden duplicate adjustment", "Included": False} for column in sorted(HR_FORBIDDEN_CONTINUOUS)]
)
covariate_audit.to_csv(TABLE_DIR / "result5_final_joint_hr_covariate_audit.csv", index=False)
joint_hr_plot.to_csv(FIGURE_DATA_DIR / "result5_final_figure6_adjusted_hr_source.csv", index=False)
display(joint_hr_plot[["Group_Label", "HR", "HR_CI_Lower", "HR_CI_Upper", "P_Value"]])
display(ph_diagnostics.head(20))


In [ ]:
# 8. Number at risk, log-rank test, and cross-panel population consistency
RISK_TIMES = np.asarray([0, 2, 4, 6, 8, 10], dtype=float)
number_at_risk_rows = []
for group in GROUP_ORDER:
    group_frame = joint_test.loc[joint_test["Group_Clean"].eq(group)]
    for time_point in RISK_TIMES:
        number_at_risk_rows.append({
            "Evaluation_Population": "test complete cases for PRS and NMR_score",
            "Group_Clean": group,
            "Group_Label": GROUP_LABELS[group],
            "Time_Years": float(time_point),
            "Number_At_Risk": int((group_frame["Time"].astype(float) >= time_point).sum()),
            "Baseline_N": int(len(group_frame)),
            "Test_Frame_SHA256": test_frame_hash,
        })
number_at_risk = pd.DataFrame(number_at_risk_rows)

number_at_risk_wide = number_at_risk.pivot(
    index=["Group_Clean", "Group_Label"],
    columns="Time_Years",
    values="Number_At_Risk",
).reset_index()
number_at_risk_wide.columns = [
    str(column) if not isinstance(column, float) else f"At_Risk_{column:g}y"
    for column in number_at_risk_wide.columns
]
number_at_risk_wide["Group_Clean"] = pd.Categorical(
    number_at_risk_wide["Group_Clean"], GROUP_ORDER, ordered=True
)
number_at_risk_wide = number_at_risk_wide.sort_values("Group_Clean")

risk_long_path = TABLE_DIR / "supplementary_table_result5_final_joint_number_at_risk.csv"
risk_wide_path = TABLE_DIR / "supplementary_table_result5_final_joint_number_at_risk_wide.csv"
number_at_risk.to_csv(risk_long_path, index=False)
number_at_risk_wide.to_csv(risk_wide_path, index=False)
try:
    number_at_risk_wide.to_excel(risk_wide_path.with_suffix(".xlsx"), index=False)
except Exception as error:
    warnings.warn(f"Optional XLSX export skipped: {error}")

logrank = multivariate_logrank_test(
    joint_test["Time"].astype(float),
    joint_test["Group_Clean"].astype(str),
    joint_test["Event"].astype(int),
)
logrank_result = pd.DataFrame([{
    "Evaluation_Population": "test complete cases for PRS and NMR_score",
    "Test": "Overall nine-group log-rank test",
    "Test_Statistic": float(logrank.test_statistic),
    "Degrees_Of_Freedom": int(len(GROUP_ORDER) - 1),
    "P_Value": float(logrank.p_value),
    "Test_Frame_SHA256": test_frame_hash,
}])
logrank_path = TABLE_DIR / "supplementary_table_result5_final_joint_logrank_test.csv"
logrank_result.to_csv(logrank_path, index=False)

expected_counts = joint_test.groupby("Group_Clean", observed=True).size().reindex(GROUP_ORDER)
absolute_counts = (
    joint_absolute.loc[joint_absolute["Horizon_Years"].eq(10.0)]
    .set_index("Group_Clean")["N"].reindex(GROUP_ORDER)
)
risk_counts = (
    number_at_risk.loc[number_at_risk["Time_Years"].eq(0.0)]
    .set_index("Group_Clean")["Number_At_Risk"].reindex(GROUP_ORDER)
)
km_counts = joint_km.groupby("Group_Clean")["N"].first().reindex(GROUP_ORDER)
population_consistency = pd.DataFrame({
    "Expected_From_Joint_Test": expected_counts,
    "Figure6A_Absolute_Risk_N": absolute_counts,
    "Figure6B_KM_N": km_counts,
    "Number_At_Risk_0y": risk_counts,
}).reset_index().rename(columns={"index": "Group_Clean"})
population_consistency["All_Equal"] = population_consistency[
    ["Expected_From_Joint_Test", "Figure6A_Absolute_Risk_N", "Figure6B_KM_N", "Number_At_Risk_0y"]
].nunique(axis=1).eq(1)
if not population_consistency["All_Equal"].all():
    raise RuntimeError("Figure 6 panels do not use identical joint-group populations.")
population_consistency.to_csv(
    FIGURE_DATA_DIR / "result5_final_figure6_population_consistency_audit.csv",
    index=False,
)
display(number_at_risk_wide)
display(logrank_result)
display(population_consistency)


In [ ]:
# 9. Figure 6 plotting helpers
KM_COLORS = {
    "PRS_low__NMR_low": "#174A8B",
    "PRS_low__NMR_mid": "#7EA6D8",
    "PRS_low__NMR_high": "#24A6B5",
    "PRS_mid__NMR_low": "#F4A582",
    "PRS_mid__NMR_mid": "#E66B4D",
    "PRS_mid__NMR_high": "#D73027",
    "PRS_high__NMR_low": "#B8A1C9",
    "PRS_high__NMR_mid": "#7B3294",
    "PRS_high__NMR_high": "#A50F15",
}

def plot_joint_heatmap(ax, data, title):
    use = data.loc[data["Horizon_Years"].eq(10.0)].copy()
    matrix = np.full((3, 3), np.nan)
    n_matrix = np.full((3, 3), np.nan)
    for _, row in use.iterrows():
        i = PRS_ORDER.index(row["PRS_Tertile"])
        j = NMR_ORDER.index(row["NMR_Tertile"])
        matrix[i, j] = float(row["Observed_KM_Risk"]) * 100
        n_matrix[i, j] = float(row["N"])
    cmap = LinearSegmentedColormap.from_list("risk_cmap", ["#EDF5FB", "#F39B7F", "#B2182B"])
    image = ax.imshow(matrix, cmap=cmap, aspect="auto")
    for i in range(3):
        for j in range(3):
            ax.text(
                j, i, f"{matrix[i, j]:.1f}%\nn={int(n_matrix[i, j]):,}",
                ha="center", va="center", fontsize=10.5, color=CNS["black"],
            )
    ax.set_xticks(range(3))
    ax.set_xticklabels([NMR_LABELS[value] for value in NMR_ORDER], rotation=20, ha="right")
    ax.set_yticks(range(3))
    ax.set_yticklabels([PRS_LABELS[value] for value in PRS_ORDER])
    ax.set_xlabel("NMR_score tertile")
    ax.set_ylabel("CAD PRS tertile")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    colorbar = plt.colorbar(image, ax=ax, fraction=0.046, pad=0.02)
    colorbar.set_label("Observed 10-year CHD risk, 1-KM (%)")

def plot_joint_km(ax, data, title):
    for group in GROUP_ORDER:
        group_data = data.loc[data["Group_Clean"].eq(group)].sort_values("Time")
        ax.step(
            group_data["Time"], group_data["Cumulative_CHD_Risk"] * 100,
            where="post", color=KM_COLORS[group],
            linewidth=2.0 if group in {"PRS_low__NMR_low", "PRS_high__NMR_high"} else 1.35,
            label=GROUP_LABELS[group],
        )
    ax.set_xlim(0, 10)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Follow-up time (years)")
    ax.set_ylabel("Kaplan-Meier cumulative CHD risk (%)")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "both")
    ax.legend(
        title="PRS tertile / NMR_score tertile", frameon=False,
        fontsize=10.5, title_fontsize=11.0, loc="upper left",
        bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, ncol=1,
    )
    p_value = float(logrank_result["P_Value"].iloc[0])
    label = "Log-rank P < 0.0001" if p_value < 1e-4 else f"Log-rank P = {p_value:.3g}"
    ax.text(0.98, 0.03, label, transform=ax.transAxes, ha="right", va="bottom", fontsize=10.5)

def plot_joint_hr(ax, data, title):
    plot = data.sort_values("Group_Clean").copy()
    y = np.arange(len(plot))
    values = plot["HR"].to_numpy(dtype=float)
    lower = plot["HR_CI_Lower"].to_numpy(dtype=float)
    upper = plot["HR_CI_Upper"].to_numpy(dtype=float)
    ax.errorbar(
        values, y,
        xerr=np.vstack([np.maximum(0, values - lower), np.maximum(0, upper - values)]),
        fmt="o", color=CNS["blue"], ecolor=CNS["blue"],
        markersize=6, markeredgecolor="white", markeredgewidth=0.6,
        elinewidth=1.2, capsize=3,
    )
    ax.axvline(1, color=CNS["gray"], linestyle="--", linewidth=1.0)
    ax.set_xscale("log")
    ax.set_yticks(y)
    ax.set_yticklabels(plot["Group_Label"])
    ax.invert_yaxis()
    ax.set_xlabel("Adjusted hazard ratio (95% CI); reference: low PRS / low NMR_score")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "x")
    handle = Line2D(
        [0], [0], marker="o", color=CNS["blue"], linewidth=1.2,
        markerfacecolor=CNS["blue"], markeredgecolor="white",
        label="Adjusted HR (95% CI)",
    )
    ax.legend(handles=[handle], frameon=False, loc="best")

def plot_joint_observed_predicted(ax, data, title):
    plot = data.loc[data["Horizon_Years"].eq(10.0)].copy()
    plot["Group_Clean"] = pd.Categorical(plot["Group_Clean"], GROUP_ORDER, ordered=True)
    plot = plot.sort_values("Group_Clean")
    y = np.arange(len(plot))
    observed = plot["Observed_KM_Risk"].to_numpy(dtype=float) * 100
    lower = plot["Observed_KM_Risk_Lower95"].to_numpy(dtype=float) * 100
    upper = plot["Observed_KM_Risk_Upper95"].to_numpy(dtype=float) * 100
    predicted = plot["Mean_Predicted_Risk"].to_numpy(dtype=float) * 100
    ax.hlines(y, np.minimum(observed, predicted), np.maximum(observed, predicted), color=CNS["light_gray"], linewidth=1.3)
    ax.errorbar(
        observed, y,
        xerr=np.vstack([np.maximum(0, observed - lower), np.maximum(0, upper - observed)]),
        fmt="o", color=CNS["red"], ecolor=CNS["red"], capsize=2.5,
        markersize=6, markeredgecolor="white", label="Observed KM risk (95% CI)",
    )
    ax.scatter(
        predicted, y, s=48, marker="s", color=CNS["blue"],
        edgecolor="white", linewidth=0.6, label="Predicted",
    )
    ax.set_yticks(y)
    ax.set_yticklabels(plot["Group_Label"])
    ax.invert_yaxis()
    ax.set_xlabel("10-year CHD risk (%)")
    ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "x")
    ax.legend(frameon=False, loc="lower right")


In [ ]:
# Figure 6A: joint 10-year cumulative-risk heatmap
fig, ax = plt.subplots(figsize=(6.9, 5.8))
plot_joint_heatmap(ax, joint_absolute, "Figure 6A. Joint 10-year CHD risk")
fig.tight_layout()
save_figure(fig, "figure6A_joint_10y_chd_risk_heatmap_cns_final")
plt.show()


In [ ]:
# Figure 6B: nine Kaplan-Meier cumulative-risk curves
fig, ax = plt.subplots(figsize=(12.6, 7.2))
plot_joint_km(ax, joint_km, "Figure 6B. Cumulative CHD risk across joint risk strata")
fig.tight_layout()
save_figure(fig, "figure6B_joint_km_cumulative_chd_risk_cns_final")
plt.show()


In [ ]:
# Figure 6C: adjusted-HR forest plot
fig, ax = plt.subplots(figsize=(8.9, 6.0))
plot_joint_hr(ax, joint_hr_plot, "Figure 6C. Adjusted hazard ratios")
fig.tight_layout()
save_figure(fig, "figure6C_joint_adjusted_hr_forest_cns_final")
plt.show()


In [ ]:
# Figure 6D: observed and predicted 10-year risk with observed 95% CI
fig, ax = plt.subplots(figsize=(9.2, 6.0))
plot_joint_observed_predicted(ax, joint_absolute, "Figure 6D. Observed and predicted 10-year CHD risk")
fig.tight_layout()
save_figure(fig, "figure6D_joint_observed_predicted_10y_risk_cns_final")
plt.show()


In [ ]:
# Output manifest and estimand audit.
figure_stems = [
    "figure5A_10y_calibration_km_ci_cns_final",
    "figure5B_10y_observed_vs_predicted_km_ci_cns_final",
    "figure5C_10y_standard_survival_dca_cns_final",
    "figure5D_10y_delta_net_benefit_survival_dca_cns_final",
    "supplementary_figure5A_5y_calibration_km_ci_cns_final",
    "supplementary_figure5B_5y_observed_vs_predicted_km_ci_cns_final",
    "supplementary_figure5C_5y_standard_survival_dca_cns_final",
    "supplementary_figure5D_5y_delta_net_benefit_survival_dca_cns_final",
    "figure6A_joint_10y_chd_risk_heatmap_cns_final",
    "figure6B_joint_km_cumulative_chd_risk_cns_final",
    "figure6C_joint_adjusted_hr_forest_cns_final",
    "figure6D_joint_observed_predicted_10y_risk_cns_final",
]
figure_paths = [
    FIGURE_DIR / f"{stem}.{extension}"
    for stem in figure_stems
    for extension in ["png", "pdf", "svg"]
]
manifest = pd.DataFrame({
    "Path": [str(path) for path in figure_paths],
    "Exists": [path.exists() for path in figure_paths],
    "Size_Bytes": [path.stat().st_size if path.exists() else np.nan for path in figure_paths],
})
manifest_path = RESULTS_OUTPUT_DIR / "result5_cns_final_separate_figure_manifest.csv"
manifest.to_csv(manifest_path, index=False)

estimand_audit = pd.DataFrame([
    {
        "Component": "Figure 5 calibration",
        "Population": "locked test set",
        "Method": "KM observed risk with Greenwood 95% CI; predictions calibrated on training only",
        "Censoring": "Handled by Kaplan-Meier",
        "Prediction_Model": RESULT5_MODEL_NORM,
    },
    {
        "Component": "Figure 5 DCA",
        "Population": "locked test set",
        "Method": "dcurves standard survival DCA",
        "Censoring": "Handled internally; no complete-case deletion and no external IPCW multiplier",
        "Prediction_Model": RESULT5_MODEL_NORM,
    },
    {
        "Component": "Figure 6A/B/D",
        "Population": "locked test-set PRS/NMR complete cases",
        "Method": "train-derived tertiles; KM observed risk; train-calibrated prediction",
        "Censoring": "Handled by Kaplan-Meier",
        "Prediction_Model": RESULT5_MODEL_NORM,
    },
    {
        "Component": "Figure 6C",
        "Population": "locked test-set PRS/NMR complete cases",
        "Method": "joint-group indicators plus clinical covariates",
        "Censoring": "Cox partial likelihood",
        "Prediction_Model": "Separate association Cox model; no continuous PRS/NMR adjustment",
    },
])
estimand_audit_path = FIGURE_DATA_DIR / "result5_final_estimand_and_model_role_audit.csv"
estimand_audit.to_csv(estimand_audit_path, index=False)

print("Manifest:", manifest_path)
print("Estimand audit:", estimand_audit_path)
print("No prediction model was retrained; Result 5 uses saved Cox PH risks only.")
print("Only train-only one-variable absolute-risk calibrators and the separate Figure 6 association Cox model were fitted.")
display(estimand_audit)
display(manifest)


In [ ]:
# Optional competing-risk sensitivity using the same training-derived tertile cutpoints as Figure 6.
from lifelines import AalenJohansenFitter
if "Competing_Event" not in df_test.columns:
    raise RuntimeError("Competing_Event is required for Result 5 sensitivity")
_competing = df_test[["Time", "Event", "Competing_Event", PRS_COL, "NMR_score"]].copy()
_competing["PRS_Tertile"] = assign_tertiles(_competing[PRS_COL], prs_edges, "PRS")
_competing["NMR_Tertile"] = assign_tertiles(_competing["NMR_score"], nmr_edges, "NMR")
_competing = _competing.loc[_competing["PRS_Tertile"].notna() & _competing["NMR_Tertile"].notna()].copy()
_competing["Group_Clean"] = _competing["PRS_Tertile"].astype(str) + "__" + _competing["NMR_Tertile"].astype(str)
if not _competing["Group_Clean"].isin(GROUP_ORDER).all():
    raise RuntimeError("Unexpected competing-risk joint group after locked cutpoint assignment")
_competing["Group_Label"] = _competing["Group_Clean"].map(GROUP_LABELS)
_competing["Event_Code"] = np.select(
    [_competing["Event"].eq(1), _competing["Competing_Event"].eq(1)], [1, 2], default=0
)
joint_curve_colors = [CNS["blue"], CNS["teal"], CNS["green"], CNS["orange"], CNS["purple"], CNS["red"], "#7E6148", "#B09C85", CNS["black"]]
aj_rows = []; fig, ax = plt.subplots(figsize=(12.8, 7.4))
for index, group in enumerate(GROUP_ORDER):
    part = _competing.loc[_competing["Group_Clean"].eq(group)]
    if part.empty:
        raise RuntimeError(f"Empty competing-risk joint group: {group}")
    label = GROUP_LABELS[group]
    estimator = AalenJohansenFitter(calculate_variance=False,seed=RANDOM_STATE + index,)
    estimator.fit(part["Time"], event_observed=part["Event_Code"], event_of_interest=1, label=label)
    curve = estimator.cumulative_density_.reset_index(); curve.columns = ["Time", "Cumulative_Incidence"]
    curve["Group_Clean"] = group; curve["Group_Label"] = label
    curve["N"] = len(part); curve["CHD_Events"] = int(part["Event"].sum())
    curve["Competing_Deaths"] = int(part["Competing_Event"].sum())
    curve["PRS_Tertile_Cut_1"] = float(prs_edges[0]); curve["PRS_Tertile_Cut_2"] = float(prs_edges[1])
    curve["NMR_Tertile_Cut_1"] = float(nmr_edges[0]); curve["NMR_Tertile_Cut_2"] = float(nmr_edges[1])
    curve["Cutpoint_Source"] = "training set; identical to primary joint-risk analysis"
    aj_rows.append(curve)
    ax.step(curve["Time"], curve["Cumulative_Incidence"], where="post", label=label,
            color=joint_curve_colors[index], alpha=0.9, linewidth=1.8)
ax.set_xlim(0, 10); ax.set_xlabel("Years since baseline"); ax.set_ylabel("CHD cumulative incidence")
ax.set_title("Competing-risk sensitivity: Aalen-Johansen", loc="left", fontweight="bold")
ax.legend(frameon=False, bbox_to_anchor=(1.02,1), loc="upper left", fontsize=10.5, ncol=1, borderaxespad=0.0); fig.tight_layout()
save_figure(fig, "supplement_result5_Aalen_Johansen_joint_groups"); plt.show()
atomic_write_csv(pd.concat(aj_rows, ignore_index=True), FIGURE_DATA_DIR/"supplement_result5_aalen_johansen_joint_groups.csv", index=False)


In [ ]:
def _pipeline_context(manifest):
    nested = manifest.get("analysis_context", {}) if isinstance(manifest, dict) else {}
    cohort_id = str(manifest.get("cohort_id", nested.get("cohort_id", "unspecified")))
    analysis_role = str(manifest.get("analysis_role", nested.get("analysis_role", "unspecified")))
    if cohort_id.lower() == "coverage90" and analysis_role.lower() == "primary":
        raise RuntimeError("coverage90 must remain a sensitivity cohort; pipeline manifest labels it primary")
    return cohort_id, analysis_role

cohort_id, analysis_role = _pipeline_context(pipeline_manifest)
context_table = pd.DataFrame([{
    "Result":"Result5", "Cohort_ID":cohort_id, "Analysis_Role":analysis_role,
    "Locked_Model":"Cox PH", "Feature_Set":PRIMARY_FULL_FEATURE_SET,
    "Joint_Tertile_Cutpoint_Source":"training set",
}])
atomic_write_csv(context_table, TABLE_DIR/"result5_analysis_context.csv", index=False)
atomic_write_json({
    "status":"success", "completed_at":str(pd.Timestamp.now()),
    "pipeline_manifest_sha256":current_pipeline_sha256(),
    "cohort_id":cohort_id, "analysis_role":analysis_role,
    "locked_model":"Cox PH", "feature_set":"Clinical_plus_PRS_plus_NMR_score",
    "selection_basis":"design lock; test not used",
    "joint_tertile_cutpoint_source":"training set",
    "competing_risk_sensitivity":"Aalen-Johansen using identical training-derived joint groups",
}, RESULT5_FINAL_DIR/"SUCCESS.json")
